In [1]:
import os, sys, glob, warnings, pickle
from collections import defaultdict
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
import scipy.stats as stats
import math

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold

from pynwb import NWBHDF5IO

# Setup Paths (Watters TensorFlow Dataset)
sys.path.append('../multi_object_memory_2025')
sys.path.append('../multi_object_memory_2025/phys_modeling')
sys.path.append('../multi_object_memory_2025/phys_modeling/training')
from phys_modeling.training.dataset import Dataset

In [2]:
# =============== #
# ENCODING MODELS #
# =============== #
class TemporalWorkingMemoryModel(nn.Module):
    def __init__(self, num_c, model_type='gain'):
        super(TemporalWorkingMemoryModel, self).__init__()
        self.baseline = nn.Parameter(torch.tensor([0.5]))
        
        # UPDATED: Direct table sizing and explicit padding for Index 0
        self.concept_tuning = nn.Embedding(num_c, 1, padding_idx=0)
        nn.init.zeros_(self.concept_tuning.weight) 
        
        if model_type == 'gain':
            self.temporal_weights = nn.Parameter(torch.ones(3))
        elif model_type == 'slot':
            self.register_buffer('temporal_weights', torch.ones(3))
            
        self.softplus = nn.Softplus()

    def forward(self, x):
        item_responses = self.concept_tuning(x).squeeze(-1)
        weighted_responses = item_responses * self.temporal_weights
        linear_rate = self.baseline + weighted_responses.sum(dim=1)
        return self.softplus(linear_rate)

class SpatialWorkingMemoryModel(nn.Module):
    def __init__(self, num_c, model_type='gain', num_positions=3):
        super(SpatialWorkingMemoryModel, self).__init__()
        self.baseline = nn.Parameter(torch.tensor([0.5]))
        
        # UPDATED: Direct table sizing and explicit padding for Index 0
        self.concept_tuning = nn.Embedding(num_c, 1, padding_idx=0)
        nn.init.zeros_(self.concept_tuning.weight)
        
        if model_type == 'gain':
            self.spatial_weights = nn.Parameter(torch.ones(num_positions))
        elif model_type == 'slot':
            self.register_buffer('spatial_weights', torch.ones(num_positions))
            
        self.softplus = nn.Softplus()

    def forward(self, x):
        item_responses = self.concept_tuning(x).squeeze(-1)
        weighted_responses = item_responses * self.spatial_weights
        linear_rate = self.baseline + weighted_responses.sum(dim=1)
        return self.softplus(linear_rate)

def inverse_softplus(mu):
    """Calculates the inverse softplus to initialize the baseline perfectly."""
    if mu < 1e-4: return -10.0 # Prevent log(0) for silent cells
    if mu > 20.0: return mu    # Prevent overflow for high-firing cells
    return math.log(math.exp(mu) - 1.0)

In [3]:
# ================ #
# CROSS-VALIDATION #
# ================ #
def evaluate_human_neuron_cv(X_trials, y_spike_counts, n_splits=5, n_epochs=500):
	X_tensor = torch.tensor(X_trials, dtype=torch.long)
	y_tensor = torch.tensor(y_spike_counts, dtype=torch.float32)
	
	# THE FIX: Dynamically calculate max index to size the embedding table safely
	num_c = int(torch.max(X_tensor).item()) + 1
	
	kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
	loss_fn = nn.PoissonNLLLoss(log_input=False, full=True)
	
	cv_nll_gain, cv_nll_slot, fold_weights = [], [], []
	
	for train_idx, test_idx in kf.split(X_tensor):
		X_train, y_train = X_tensor[train_idx], y_tensor[train_idx]
		X_test, y_test   = X_tensor[test_idx], y_tensor[test_idx]
		
		# Train Gain
		gain_model = TemporalWorkingMemoryModel(num_c, model_type='gain')
		mean_spikes = float(np.mean(y_train.numpy()))
		gain_model.baseline.data = torch.tensor([inverse_softplus(mean_spikes)])
		# gain_model.baseline.data = torch.tensor([float(np.mean(y_train.numpy()))])
		optimizer_g = optim.Adam(gain_model.parameters(), lr=0.01)
		for _ in range(n_epochs):
			optimizer_g.zero_grad()
			loss_fn(gain_model(X_train), y_train).backward()
			optimizer_g.step()
			
		fold_weights.append(gain_model.temporal_weights.detach().numpy())
			
		with torch.no_grad():
			cv_nll_gain.append(loss_fn(gain_model(X_test), y_test).item())
			
		# Train Slot
		slot_model = TemporalWorkingMemoryModel(num_c, model_type='slot')
		mean_spikes = float(np.mean(y_train.numpy()))
		slot_model.baseline.data = torch.tensor([inverse_softplus(mean_spikes)])
		# slot_model.baseline.data = torch.tensor([float(np.mean(y_train.numpy()))])
		optimizer_s = optim.Adam(slot_model.parameters(), lr=0.01)
		for _ in range(n_epochs):
			optimizer_s.zero_grad()
			loss_fn(slot_model(X_train), y_train).backward()
			optimizer_s.step()
			
		with torch.no_grad():
			cv_nll_slot.append(loss_fn(slot_model(X_test), y_test).item())

	return np.mean(cv_nll_gain), np.mean(cv_nll_slot), np.mean(fold_weights, axis=0)

def evaluate_macaque_neuron_cv(X_trials_spatial, y_spike_counts, num_positions=3, n_splits=5, n_epochs=500):
	X_tensor = torch.tensor(X_trials_spatial, dtype=torch.long)
	y_tensor = torch.tensor(y_spike_counts, dtype=torch.float32)
	
	# THE FIX: Dynamically calculate max index to size the embedding table safely
	num_c = int(torch.max(X_tensor).item()) + 1
	
	kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
	loss_fn = nn.PoissonNLLLoss(log_input=False, full=True)
	
	cv_nll_gain, cv_nll_slot = [], []
	fold_weights = [] 
	
	for train_idx, test_idx in kf.split(X_tensor):
		X_train, y_train = X_tensor[train_idx], y_tensor[train_idx]
		X_test, y_test   = X_tensor[test_idx], y_tensor[test_idx]
		
		# Train Gain
		gain_model = SpatialWorkingMemoryModel(num_c, model_type='gain', num_positions=num_positions)
		gain_model.baseline.data = torch.tensor([float(np.mean(y_train.numpy()))])
		opt_g = optim.Adam(gain_model.parameters(), lr=0.01)
		for _ in range(n_epochs):
			opt_g.zero_grad(); loss_fn(gain_model(X_train), y_train).backward(); opt_g.step()
			
		fold_weights.append(gain_model.spatial_weights.detach().numpy())
			
		with torch.no_grad(): 
			cv_nll_gain.append(loss_fn(gain_model(X_test), y_test).item())
			
		# Train Slot
		slot_model = SpatialWorkingMemoryModel(num_c, model_type='slot', num_positions=num_positions)
		slot_model.baseline.data = torch.tensor([float(np.mean(y_train.numpy()))])
		opt_s = optim.Adam(slot_model.parameters(), lr=0.01)
		for _ in range(n_epochs):
			opt_s.zero_grad(); loss_fn(slot_model(X_train), y_train).backward(); opt_s.step()
			
		with torch.no_grad(): 
			cv_nll_slot.append(loss_fn(slot_model(X_test), y_test).item())

	return np.mean(cv_nll_gain), np.mean(cv_nll_slot), np.mean(fold_weights, axis=0)

In [4]:
# ============== #
# HUMAN MODELING #
# ============== #
def format_human_data_for_modeling(human_data_root, custom_cache_dir, target_regions):
	"""
	Parse NWB files and Concept Cell CSVs to build the trial-by-trial dataset.
	Returns:
	human_cells_dict: Dictionary formatted as:
		{ 'sub-X_ses-Y_unitZ_region': (X_trials_array, y_spikes_array) }
	"""
	human_cells_dict = {}
	
	# 1. Find all concept cell CSVs
	csv_pattern = os.path.join(custom_cache_dir, "*_concept_cells.csv")
	csv_files = glob.glob(csv_pattern)
	print(f"Found {len(csv_files)} concept cell CSV files.")
	for csv_path in csv_files:
		# Load the CSV
		df_concepts = pd.read_csv(csv_path)
		# Filter for our target frontal regions
		df_target = df_concepts[df_concepts['location'].isin(target_regions)]
		if df_target.empty:
			continue
		# 2. Extract Subject and Session from the filename
		# Ex: 'sub-2_ses-2_ecephys+image_concept_cells.csv'
		filename = os.path.basename(csv_path)
		parts = filename.split('_')
		sub_id = parts[0] # 'sub-2'
		ses_id = parts[1] # 'ses-2'
		# 3. Locate the corresponding Working Memory NWB file
		# We use a flexible search in case the exact sub-folder structure varies
		nwb_search_pattern = os.path.join(human_data_root, "**", f"{sub_id}_{ses_id}*WorkingMemory*.nwb")
		nwb_matches = glob.glob(nwb_search_pattern, recursive=True)
		
		# Fallback if task name varies slightly
		if not nwb_matches:
			nwb_search_pattern = os.path.join(human_data_root, "**", f"{sub_id}_{ses_id}*.nwb")
			nwb_matches = glob.glob(nwb_search_pattern, recursive=True)
		if not nwb_matches:
			print(f"Warning: Could not find NWB file for {sub_id} {ses_id}. Skipping.")
			continue
			
		nwb_path = nwb_matches[0]
		
		# 4. Open the NWB file and extract data
		with NWBHDF5IO(nwb_path, 'r') as io:
			nwb = io.read()
			trials_df = nwb.trials.to_dataframe()
			# Clean trials: drop aborted trials where Maintenance didn't happen
			trials_df = trials_df.dropna(subset=['timestamps_Maintenance', 'timestamps_Probe'])
			# 5. Extract data for each valid target cell
			for _, cell_row in df_target.iterrows():
				unit_id = int(cell_row['unit_id'])
				location = cell_row['location']
				# Fetch spike times for this specific unit
				# (PyNWB usually indexes units via get_unit_spike_times)
				try:
					spike_times = nwb.units.get_unit_spike_times(unit_id)
				except:
					print(f"Warning: Unit {unit_id} not found in {nwb_path}. Skipping.")
					continue
				
				X_trials = []
				y_spikes = []
				
				# Iterate through trials to build X and y
				for _, trial in trials_df.iterrows():
					# 5a. Count Spikes in the Delay Period
					delay_start = trial['timestamps_Maintenance']
					delay_end = trial['timestamps_Probe']
					# Fast numpy approach to count spikes within the window
					spikes_in_delay = np.sum((spike_times >= delay_start) & (spike_times < delay_end))
					# 5b. Build the Temporal Sequence Array (Item 1, Item 2, Item 3)
					# NOTE: Ensure these column names match your NWB's trial table 
					# (e.g., 'image_1', 'stimulus_1', etc.)
					load = int(trial['loads'])
					# Default padding is 0
					seq = [0, 0, 0] 
					if load >= 1:
						seq[0] = int(trial['loadsEnc1_PicIDs'])
					if load >= 2:
						seq[1] = int(trial['loadsEnc2_PicIDs'])
					if load >= 3:
						seq[2] = int(trial['loadsEnc3_PicIDs'])
					
					X_trials.append(seq)
					y_spikes.append(spikes_in_delay)
				
				# 6. Save to Dictionary with Anatomical Tag
				cell_key = f"{sub_id}_{ses_id}_unit{unit_id}_{location}"
				human_cells_dict[cell_key] = (np.array(X_trials), np.array(y_spikes))

	print(f"Extraction complete! Formatted {len(human_cells_dict)} frontal concept cells.")
	return human_cells_dict

def evaluate_population_by_region(human_cells_dict):
	delta_nll_all, nll_gain_all, nll_slot_all = [], [], []
	region_preferences = defaultdict(lambda: {'Gain': 0, 'Slot': 0, 'Indifferent': 0})
	region_deltas = defaultdict(list)
	gain_cell_weights = []
	
	print(f"Beginning evaluation of {len(human_cells_dict)} concept cells...")
	for i, (cell_key, (X_trials, y_spike_counts)) in enumerate(human_cells_dict.items()):
		
		if 'dorsal_anterior_cingulate_cortex' in cell_key: broad_region = 'dACC'
		elif 'pre_supplementary_motor_area' in cell_key: broad_region = 'pre-SMA'
		else: broad_region = 'Other'
			
		# Removed num_concepts argument
		cv_gain, cv_slot, weights = evaluate_human_neuron_cv(X_trials, y_spike_counts, n_splits=5)
		nll_gain_all.append(cv_gain); nll_slot_all.append(cv_slot)
		
		delta = cv_slot - cv_gain
		delta_nll_all.append(delta)
		region_deltas[broad_region].append(delta)
		
		if delta > 0.01:
			region_preferences[broad_region]['Gain'] += 1
			gain_cell_weights.append(weights)
		elif delta < -0.01: region_preferences[broad_region]['Slot'] += 1
		else: region_preferences[broad_region]['Indifferent'] += 1

	# --- Print Regional Breakdown ---
	print("\n" + "="*50)
	print("ANATOMICAL DISSOCIATION: GAIN VS SLOT")
	print("="*50)
	
	for region, prefs in region_preferences.items():
		total = prefs['Gain'] + prefs['Slot'] + prefs['Indifferent']
		if total == 0: continue
			
		print(f"\nRegion: {region} (n={total} cells)")
		print(f"  Gain-Preferring: {prefs['Gain']} ({(prefs['Gain']/total)*100:.1f}%)")
		print(f"  Slot-Preferring: {prefs['Slot']} ({(prefs['Slot']/total)*100:.1f}%)")
		print(f"  Indifferent:     {prefs['Indifferent']} ({(prefs['Indifferent']/total)*100:.1f}%)")
		
		if total >= 10:
			stat, p_val = stats.wilcoxon(region_deltas[region])
			median_delta = np.median(region_deltas[region])
			significance = "*" if p_val < 0.05 else "ns"
			print(f"  Wilcoxon p-val:  {p_val:.4e} ({significance}) | Median ΔNLL = {median_delta:.3f}")
		else:
			print("  [Sample size too small for rigorous Wilcoxon test]")

	# --- Print Population Statistics ---
	stat_pop, p_val_pop = stats.wilcoxon(nll_gain_all, nll_slot_all)
	median_pop = np.median(delta_nll_all)
	
	print("\n" + "="*50)
	print("GLOBAL POPULATION STATISTICS")
	print("="*50)
	print(f"Median Global ΔNLL: {median_pop:.4f}")
	print(f"Global Wilcoxon p:  {p_val_pop:.4e}")

	return dict(region_preferences), dict(region_deltas), delta_nll_all, np.array(gain_cell_weights)

In [5]:
# ================== #
# RUN HUMAN PIPELINE #
# ================== #
human_data_root = "/Volumes/fetty/000469"
custom_cache_dir = "/Users/cwook/Documents/humac/data/concept_cells"
human_target_regions = [
	'pre_supplementary_motor_area_right', 
	'pre_supplementary_motor_area_left', 
	'dorsal_anterior_cingulate_cortex_right',
	'dorsal_anterior_cingulate_cortex_left'
]

human_cells_dict = format_human_data_for_modeling(human_data_root, custom_cache_dir, human_target_regions)
human_prefs, human_deltas, human_all_deltas, human_weights_array = evaluate_population_by_region(human_cells_dict)
human_pred_out = {
	'human_cells_dict': human_cells_dict,
	'human_all_deltas': human_all_deltas,
	'human_deltas': human_deltas,
	'human_prefs': human_prefs,
	'human_weights_array': human_weights_array
	}

# Save output
with open('/Users/cwook/Documents/humac/output/human_pred_out.pickle', 'wb') as handle:
	pickle.dump(human_pred_out, handle, protocol=pickle.HIGHEST_PROTOCOL)

Found 17 concept cell CSV files.
Extraction complete! Formatted 17 frontal concept cells.
Beginning evaluation of 17 concept cells...

ANATOMICAL DISSOCIATION: GAIN VS SLOT

Region: pre-SMA (n=11 cells)
  Gain-Preferring: 0 (0.0%)
  Slot-Preferring: 8 (72.7%)
  Indifferent:     3 (27.3%)
  Wilcoxon p-val:  2.9297e-03 (*) | Median ΔNLL = -0.054

Region: dACC (n=6 cells)
  Gain-Preferring: 2 (33.3%)
  Slot-Preferring: 4 (66.7%)
  Indifferent:     0 (0.0%)
  [Sample size too small for rigorous Wilcoxon test]

GLOBAL POPULATION STATISTICS
Median Global ΔNLL: -0.0282
Global Wilcoxon p:  4.6387e-03


In [6]:
# ================ #
# MACAQUE MODELING #
# ================ #
def strict_watters_filter(df):
	clean_units = df['quality'].isin(['good', '1', 'sua'])
	pos_sig = (df['Position AUC'] > 0.65) & (df['Position p-value'] < 0.05)
	id_sig = (df['Identity AUC'] > 0.65) & (df['Identity p-value'] < 0.05)
	is_probeloc = df['probe'] == 's0' 
	return df[clean_units & (pos_sig | id_sig) & is_probeloc]

def extract_macaque_trial_by_trial(session_list):
	print(f"\nExtracting Trial-by-Trial Data for {len(session_list)} sessions...")
	macaque_cells_dict = {}
	
	# 0 = empty screen position. 1, 2, 3 = objects
	concept_map = {'a': 1, 'b': 2, 'c': 3}
	
	for subject, session in session_list:
		try:
			ds_delay = Dataset(subject=subject, session=session, phase="delay", 
							   unit_filter=strict_watters_filter, 
							   trial_filter=lambda df: df, max_n_objects=3)
							   
			raw_neural = ds_delay.data['neural'] 
			if raw_neural.shape[2] == 0: 
				continue
				
			total_delay_spikes = np.sum(raw_neural, axis=1)
			
			# --- THE FIX: Aggressively Hunt for the DataFrame ---
			trials_df = None
			# 1. Check inside the .data dictionary first
			if isinstance(ds_delay.data, dict):
				for key, val in ds_delay.data.items():
					if isinstance(val, pd.DataFrame):
						trials_df = val
						break       
			# 2. If not found, scan all public attributes of the object
			if trials_df is None:
				for attr in dir(ds_delay):
					if not attr.startswith('_'):
						try:
							val = getattr(ds_delay, attr)
							if isinstance(val, pd.DataFrame):
								trials_df = val
								break
						except Exception:
							pass
							
			if trials_df is None:
				raise AttributeError("Could not find a pandas DataFrame anywhere inside the Dataset object.")
				
			# 3. Verify alignment (Crucial for trial-by-trial modeling)
			if len(trials_df) != raw_neural.shape[0]:
				print(f" -> Warning on {session}: DataFrame rows ({len(trials_df)}) do not match Neural trials ({raw_neural.shape[0]}).")
				# If this happens, it usually means we need to filter for completed trials
				trials_df = trials_df[trials_df['completed'] == True]
			# ----------------------------------------------------
			
			X_trials = []
			# Iterate through the found dataframe
			for _, row in trials_df.iterrows():
				spatial_array = [0, 0, 0]
				load = int(row['num_objects'])
				if load >= 1 and pd.notna(row['object_0_location']):
					spatial_array[int(row['object_0_location'])] = concept_map[row['object_0_id']]
				if load >= 2 and pd.notna(row['object_1_location']):
					spatial_array[int(row['object_1_location'])] = concept_map[row['object_1_id']]
				if load >= 3 and pd.notna(row['object_2_location']):
					spatial_array[int(row['object_2_location'])] = concept_map[row['object_2_id']]
					
				X_trials.append(spatial_array)
				
			X_trials = np.array(X_trials)
			n_neurons = raw_neural.shape[2]
			for neuron_idx in range(n_neurons):
				y_spikes = total_delay_spikes[:, neuron_idx]
				cell_key = f"{subject}_{session}_unit{neuron_idx}"
				macaque_cells_dict[cell_key] = (X_trials, y_spikes)
				
			print(f" -> {session} extracted ({n_neurons} DMFC neurons)")
			
		except Exception as e:
			print(f" -> Error on {session}: {e}")
			
	print(f"\nExtraction complete! Formatted {len(macaque_cells_dict)} Macaque DMFC cells.")
	return macaque_cells_dict

def evaluate_macaque_population(macaque_dataset_dict, num_positions=3):
	mac_deltas = []
	mac_prefs = {'Gain': 0, 'Slot': 0, 'Indifferent': 0}
	gain_cell_spatial_weights = [] 
	print(f"Beginning evaluation of {len(macaque_dataset_dict)} macaque DMFC cells...")
	for i, (cell_key, (X_trials, y_spikes)) in enumerate(macaque_dataset_dict.items()):
		
		# Removed num_concepts argument
		cv_gain, cv_slot, weights = evaluate_macaque_neuron_cv(X_trials, y_spikes, num_positions=num_positions)
		delta = cv_slot - cv_gain
		mac_deltas.append(delta)
		if delta > 0.01:
			mac_prefs['Gain'] += 1
			gain_cell_spatial_weights.append(weights)
		elif delta < -0.01: 
			mac_prefs['Slot'] += 1
		else: 
			mac_prefs['Indifferent'] += 1

	total_macaque_cells = sum(mac_prefs.values())
	gain_pct = (mac_prefs['Gain'] / total_macaque_cells) * 100
	slot_pct = (mac_prefs['Slot'] / total_macaque_cells) * 100
	indiff_pct = (mac_prefs['Indifferent'] / total_macaque_cells) * 100
	median_delta = np.median(mac_deltas)

	stat, p_val = stats.wilcoxon(mac_deltas)

	print("="*50)
	print("MACAQUE DMFC POPULATION STATISTICS (SPATIAL)")
	print("="*50)
	print(f"Total Cells Analyzed: {total_macaque_cells}")
	print(f"Gain-Preferring:      {mac_prefs['Gain']} ({gain_pct:.1f}%)")
	print(f"Slot-Preferring:      {mac_prefs['Slot']} ({slot_pct:.1f}%)")
	print(f"Indifferent:          {mac_prefs['Indifferent']} ({indiff_pct:.1f}%)")
	print(f"Median ΔNLL:          {median_delta:.4f}")
	print(f"Wilcoxon p-val:       {p_val:.4e}")
	return mac_deltas, mac_prefs, np.array(gain_cell_spatial_weights)

In [7]:
# ==================== #
# RUN MACAQUE PIPELINE #
# ==================== #
mac_shape = pd.read_csv('/Users/cwook/Documents/humac/lists/task_shape_inventory.csv')
macaque_sessions = []
for n,r in mac_shape[mac_shape['shape'] == 'Triangle'].iterrows():
	sub = r['file'].split('/')[0].replace('sub-','')
	ses = r['file'].split('/')[1].split('_')[1].replace('ses-','')
	macaque_sessions.append((sub,ses))
macaque_sessions = macaque_sessions
macaque_cells_dict = extract_macaque_trial_by_trial(macaque_sessions)
macaque_deltas, macaque_prefs, macaque_weights_array = evaluate_macaque_population(macaque_cells_dict, num_positions=3)
macaque_pred_out = {
	'macaque_cells_dict': macaque_cells_dict,
	'macaque_deltas': macaque_deltas,
	'macaque_prefs': macaque_prefs,
	'macaque_weights_array': macaque_weights_array
	}

# Save output
with open('/Users/cwook/Documents/humac/output/macaque_pred_out.pickle', 'wb') as handle:
	pickle.dump(macaque_pred_out, handle, protocol=pickle.HIGHEST_PROTOCOL)


Extracting Trial-by-Trial Data for 24 sessions...
Triangle session
Original number of trials: 2267
Original number of units: 203
Number of trials after filtering: 0
Number of units after filtering: 0
 -> Error on 2022-09-03: No units left after filtering
Triangle session
Original number of trials: 1512
Original number of units: 151
Number of trials after filtering: 1193
Number of units after filtering: 10
 -> 2022-09-02 extracted (10 DMFC neurons)
Triangle session
Original number of trials: 1578
Original number of units: 206
Number of trials after filtering: 1238
Number of units after filtering: 26
 -> 2022-08-31 extracted (26 DMFC neurons)
Triangle session
Original number of trials: 2847
Original number of units: 323
Number of trials after filtering: 1253
Number of units after filtering: 31
 -> 2022-08-19 extracted (31 DMFC neurons)
Triangle session
Original number of trials: 1954
Original number of units: 290
Number of trials after filtering: 1336
Number of units after filtering: 17

In [8]:
# Fisher's Exact Test
contingency_table = np.array([[13, 4], [271, 73]])
odds_ratio, p_val_fisher = stats.fisher_exact(contingency_table)
print(f"Fisher's Exact p-value: {p_val_fisher:.4f}")

# Wilcoxon Effect Sizes
def calc_wilcoxon_effect_size(p_val, n_samples):
	return np.abs(stats.norm.ppf(p_val / 2)) / np.sqrt(n_samples)

print(f"Human Effect Size (r):   {calc_wilcoxon_effect_size(1.6785e-03, 17):.3f}")
print(f"Macaque Effect Size (r): {calc_wilcoxon_effect_size(2.6421e-44, 533):.3f}")

Fisher's Exact p-value: 0.7664
Human Effect Size (r):   0.762
Macaque Effect Size (r): 0.605
